# 🔬 Custom OOD Test — 16K Model (java-vuln-adapter-32b-full)

This notebook tests the fine-tuned model on **out-of-distribution (OOD)** code — patterns the model likely hasn't seen verbatim in the 16K training rows.

## Test Strategy
| # | Test Name | Why it's OOD / Hard |
|---|-----------|----------------------|
| 1 | **SQL Injection via String.format** | Training used `+` concatenation; this uses `String.format()` — different syntax pattern |
| 2 | **XSS via JSON response body** | Training XSS was HTML rendering; this is REST API returning unsanitised JSON |
| 3 | **Command Injection via ProcessBuilder** | Training used `Runtime.exec(String)`; this uses `ProcessBuilder` with shell wrapper |
| 4 | **Path Traversal: Zip Slip** | Zip-slip variant — specific sub-type rarely seen in training data |
| 5 | **Insecure Deserialization (ObjectInputStream)** | Very underrepresented type (only 26 samples in 16K rows) |
| 6 | **Subtle SQL Injection (multi-param WHERE)** | Original Custom_Test template case — dynamic WHERE building |
| 7 | **CLEAN safe code** | Model should correctly say it's secure (false-positive check) |
| 8 | **Mixed: SQL + Log Injection combo** | Two vulnerabilities in one class — training had single-vuln examples |

In [ ]:
# ─── Installation Cell ───
# Removed: No longer needed for ROCm environment.

In [ ]:
# ═══ CELL 1: Load Model Once ════════════════════════════════════════════════
import sys
import torch
from unittest.mock import MagicMock

# 1. BYPASS PYTORCH BUG: Mock deepgemm to prevent torch._dynamo import error
sys.modules['transformers.integrations.deepgemm'] = MagicMock()

# 2. Clear cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_ID     = "Qwen/Qwen2.5-Coder-32B-Instruct"
ADAPTER_PATH = "./java-vuln-adapter-32b-full"

print(f"Loading base model {MODEL_ID}...")
tokenizer  = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, 
    device_map="auto", 
    torch_dtype=torch.bfloat16
)

print(f"Applying adapter {ADAPTER_PATH}...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

from peft.tuners.lora.layer import LoraLayer
for module in model.modules():
    if isinstance(module, LoraLayer):
        module.set_scale("default", 0.8)

model.config.use_cache = True
model.eval()

DEVICE = next(model.parameters()).device
print(f"✅ Model ready on {DEVICE} (Adapter scale=0.8) — {torch.cuda.memory_allocated(0)/1e9:.1f} GB VRAM used")


In [ ]:
# ═══ CELL 2: Shared Inference Helper ═══════════════════════════════════════
import sys

SYSTEM_PROMPT = """You are an expert Java security auditor. Analyze the provided code.
If the code is secure, output:
\"This Java code is completely secure and contains no vulnerabilities. No changes are required.\"

If the code is vulnerable, output your analysis in this exact format:
### 🛡️ Vulnerability Analysis
*   **Status**: VULNERABLE
*   **Type**: [Vulnerability Type]
*   **Severity**: HIGH

### 📝 Explanation
[Provide a brief explanation of the vulnerability]

### 🛠️ Fixed Code
```java
[Fixed complete Java code]
```"""

def analyze(code: str, label: str = "") -> str:
    """Run the model on a code snippet and print structured output."""
    model.generation_config.max_new_tokens = 1200
    model.generation_config.max_length     = None

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Analyze the following Java code. If a vulnerability exists, provide the fixed code. If it is safe, output the original code.\n\n{code}"}
    ]
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)

    print(f"\n{'═'*70}")
    print(f"🔍  {label}")
    print(f"{'═'*70}")
    print("Analyzing... (wait ~30s)")
    sys.stdout.flush()

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=1200,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id
        )

    gen_tokens = out[0][inputs.input_ids.shape[1]:].cpu()
    result = tokenizer.decode(gen_tokens, skip_special_tokens=True)

    print(f"Done — {len(gen_tokens)} tokens generated.")
    print(f"\n{'─'*70}")
    print(result)
    print(f"{'─'*70}")
    return result

print("✅ Helper ready. Run tests below.")

---
## 🔴 TEST 1 — SQL Injection via `String.format()` (OOD: unseen syntax)

**Why this is OOD**: Training data used `+` string concatenation to build queries. This uses `String.format()` — functionally identical vulnerability but different lexical pattern the model may not recognise.

**Expected**: VULNERABLE → SQL Injection

**If model fails**: It may say SECURE or miss the injection entirely.

**Fixed version**: Uses `PreparedStatement` with `?` placeholders instead of `String.format`.

In [ ]:
# ─── VULNERABLE CODE ─────────────────────────────────────────────────────────
TEST1_VULNERABLE = """
import java.sql.Connection;
import java.sql.ResultSet;
import java.sql.SQLException;
import java.sql.Statement;

public class ProductSearchService {

    private final Connection conn;

    public ProductSearchService(Connection conn) {
        this.conn = conn;
    }

    // Searches products by category and minimum price
    public ResultSet search(String category, double minPrice) throws SQLException {
        String sql = String.format(
            "SELECT * FROM products WHERE category = '%s' AND price >= %.2f",
            category, minPrice
        );
        Statement stmt = conn.createStatement();
        return stmt.executeQuery(sql);
    }
}
"""

# ─── FIXED CODE (reference) ───────────────────────────────────────────────────
TEST1_FIXED = """
import java.sql.Connection;
import java.sql.PreparedStatement;
import java.sql.ResultSet;
import java.sql.SQLException;

public class ProductSearchService {

    private final Connection conn;

    public ProductSearchService(Connection conn) {
        this.conn = conn;
    }

    // FIXED: PreparedStatement with parameterised placeholders — no injection possible
    public ResultSet search(String category, double minPrice) throws SQLException {
        String sql = "SELECT * FROM products WHERE category = ? AND price >= ?";
        PreparedStatement stmt = conn.prepareStatement(sql);
        stmt.setString(1, category);
        stmt.setDouble(2, minPrice);
        return stmt.executeQuery();
    }
}
"""

print("=" * 60)
print("TEST 1 — REFERENCE FIXED CODE")
print("=" * 60)
print(TEST1_FIXED)

result1 = analyze(TEST1_VULNERABLE, "TEST 1 | SQL Injection via String.format() — OOD Syntax")

---
## 🔴 TEST 2 — XSS via REST API / JSON Response (OOD: API context)

**Why this is OOD**: Training XSS was HTML template rendering. This is a REST endpoint returning user data directly in a JSON body — same class of vulnerability but in a completely different architectural context.

**Expected**: VULNERABLE → XSS / Injection

**If model fails**: It may miss it because it's not HTML rendering.

**Fixed version**: Escapes HTML meta-characters before embedding in JSON output + sets security headers.

In [ ]:
# ─── VULNERABLE CODE ─────────────────────────────────────────────────────────
TEST2_VULNERABLE = """
import javax.servlet.http.HttpServletRequest;
import javax.servlet.http.HttpServletResponse;
import java.io.IOException;
import java.io.PrintWriter;

public class ProfileApiServlet extends javax.servlet.http.HttpServlet {

    @Override
    protected void doGet(HttpServletRequest req, HttpServletResponse resp)
            throws IOException {

        // API returns user's display name in JSON response
        String displayName = req.getParameter("name");   // raw, unsanitised
        String bio         = req.getParameter("bio");    // raw, unsanitised

        resp.setContentType("application/json");
        PrintWriter out = resp.getWriter();

        // Embedding unsanitised HTML/script into JSON value —
        // if the frontend uses innerHTML to render this, it triggers XSS
        out.println("{");
        out.println("  \"name\": \"" + displayName + "\",");
        out.println("  \"bio\": \""  + bio         + "\"");
        out.println("}");
    }
}
"""

# ─── FIXED CODE (reference) ───────────────────────────────────────────────────
TEST2_FIXED = """
import javax.servlet.http.HttpServletRequest;
import javax.servlet.http.HttpServletResponse;
import java.io.IOException;
import java.io.PrintWriter;

public class ProfileApiServlet extends javax.servlet.http.HttpServlet {

    // FIXED: Escape special HTML/JSON chars before embedding in output
    private static String escapeJson(String value) {
        if (value == null) return "";
        return value
            .replace("\\\\", "\\\\\\\\")
            .replace("\"", "\\\\\"")
            .replace("<",  "\\u003C")
            .replace(">",  "\\u003E")
            .replace("&",  "\\u0026")
            .replace("'",  "\\u0027");
    }

    @Override
    protected void doGet(HttpServletRequest req, HttpServletResponse resp)
            throws IOException {

        String displayName = req.getParameter("name");
        String bio         = req.getParameter("bio");

        resp.setContentType("application/json; charset=UTF-8");
        resp.setHeader("X-Content-Type-Options", "nosniff"); // added security header
        PrintWriter out = resp.getWriter();

        // FIXED: Sanitise before embedding in response
        out.println("{");
        out.println("  \"name\": \"" + escapeJson(displayName) + "\",");
        out.println("  \"bio\": \""  + escapeJson(bio)         + "\"");
        out.println("}");
    }
}
"""

print("=" * 60)
print("TEST 2 — REFERENCE FIXED CODE")
print("=" * 60)
print(TEST2_FIXED)

result2 = analyze(TEST2_VULNERABLE, "TEST 2 | XSS in REST API JSON Response — OOD Context")

---
## 🔴 TEST 3 — Command Injection via `ProcessBuilder` + shell wrapper (OOD)

**Why this is OOD**: Training data used `Runtime.exec(String)` patterns. This uses `ProcessBuilder` with `["sh", "-c", ...]` — the shell wrapper makes it exploitable even though `ProcessBuilder` is normally considered safer.

**Expected**: VULNERABLE → Command Injection

**If model fails**: May say SECURE because it sees `ProcessBuilder` (not `Runtime.exec`).

**Fixed version**: Validate input against an allowlist + pass args as separate tokens (no shell wrapper).

In [ ]:
# ─── VULNERABLE CODE ─────────────────────────────────────────────────────────
TEST3_VULNERABLE = """
import java.io.BufferedReader;
import java.io.InputStreamReader;
import java.util.Arrays;
import java.util.List;

public class PingService {

    /**
     * Pings a host and returns the output.
     * Uses ProcessBuilder (considered safer than Runtime.exec) but still vulnerable
     * because the sh -c wrapper passes user input directly to the shell.
     * Attacker input example: host = "google.com; rm -rf /tmp/important"
     */
    public String pingHost(String host) throws Exception {
        List<String> cmd = Arrays.asList("sh", "-c", "ping -c 1 " + host);
        ProcessBuilder pb = new ProcessBuilder(cmd);
        pb.redirectErrorStream(true);

        Process proc = pb.start();
        BufferedReader reader = new BufferedReader(
            new InputStreamReader(proc.getInputStream()));

        StringBuilder sb = new StringBuilder();
        String line;
        while ((line = reader.readLine()) != null) {
            sb.append(line).append("\n");
        }
        return sb.toString();
    }
}
"""

# ─── FIXED CODE (reference) ───────────────────────────────────────────────────
TEST3_FIXED = """
import java.io.BufferedReader;
import java.io.InputStreamReader;
import java.util.regex.Pattern;

public class PingService {

    // FIXED 1: Allowlist — only valid hostname/IP characters permitted
    private static final Pattern SAFE_HOST = Pattern.compile("^[a-zA-Z0-9.\\-]{1,253}$");

    public String pingHost(String host) throws Exception {
        // FIXED 2: Validate input before use
        if (host == null || !SAFE_HOST.matcher(host).matches()) {
            throw new IllegalArgumentException("Invalid host: " + host);
        }

        // FIXED 3: Pass args as separate tokens — NO shell wrapper (no sh -c)
        ProcessBuilder pb = new ProcessBuilder("ping", "-c", "1", host);
        pb.redirectErrorStream(true);

        Process proc = pb.start();
        BufferedReader reader = new BufferedReader(
            new InputStreamReader(proc.getInputStream()));

        StringBuilder sb = new StringBuilder();
        String line;
        while ((line = reader.readLine()) != null) {
            sb.append(line).append("\n");
        }
        return sb.toString();
    }
}
"""

print("=" * 60)
print("TEST 3 — REFERENCE FIXED CODE")
print("=" * 60)
print(TEST3_FIXED)

result3 = analyze(TEST3_VULNERABLE, "TEST 3 | Command Injection via ProcessBuilder+shell — OOD Pattern")

---
## 🔴 TEST 4 — Path Traversal: Zip Slip (OOD: specific sub-type)

**Why this is OOD**: Only 104 Path Traversal samples exist in 16K rows. Zip Slip is a specific variant — writing extracted zip entries to the filesystem without canonicalising the path — that the model almost certainly hasn't seen.

**Expected**: VULNERABLE → Path Traversal (Zip Slip)

**If model fails**: May miss it entirely or output a wrong vulnerability type.

**Fixed version**: Canonicalise and verify the resolved path stays inside the destination directory.

In [ ]:
# ─── VULNERABLE CODE ─────────────────────────────────────────────────────────
TEST4_VULNERABLE = """
import java.io.*;
import java.util.zip.ZipEntry;
import java.util.zip.ZipInputStream;

public class ZipExtractor {

    private static final int BUFFER = 4096;

    /**
     * Extracts a ZIP archive to the destination directory.
     * Vulnerable to Zip Slip: a crafted archive entry like
     * ../../etc/cron.d/evil can escape the destination directory.
     */
    public void extract(InputStream zipStream, File destDir) throws IOException {
        try (ZipInputStream zis = new ZipInputStream(zipStream)) {
            ZipEntry entry;
            while ((entry = zis.getNextEntry()) != null) {
                // No path validation — entry.getName() may contain ../
                File outFile = new File(destDir, entry.getName());

                if (entry.isDirectory()) {
                    outFile.mkdirs();
                } else {
                    outFile.getParentFile().mkdirs();
                    try (BufferedOutputStream bos =
                             new BufferedOutputStream(new FileOutputStream(outFile), BUFFER)) {
                        byte[] buf = new byte[BUFFER];
                        int len;
                        while ((len = zis.read(buf)) > 0) {
                            bos.write(buf, 0, len);
                        }
                    }
                }
                zis.closeEntry();
            }
        }
    }
}
"""

# ─── FIXED CODE (reference) ───────────────────────────────────────────────────
TEST4_FIXED = """
import java.io.*;
import java.util.zip.ZipEntry;
import java.util.zip.ZipInputStream;

public class ZipExtractor {

    private static final int BUFFER = 4096;

    public void extract(InputStream zipStream, File destDir) throws IOException {
        String canonicalDest = destDir.getCanonicalPath();

        try (ZipInputStream zis = new ZipInputStream(zipStream)) {
            ZipEntry entry;
            while ((entry = zis.getNextEntry()) != null) {
                File outFile = new File(destDir, entry.getName());

                // FIXED: Canonicalise and verify the resolved path stays inside destDir
                String canonicalOut = outFile.getCanonicalPath();
                if (!canonicalOut.startsWith(canonicalDest + File.separator)) {
                    throw new IOException("Zip Slip detected — entry escapes dest: " + entry.getName());
                }

                if (entry.isDirectory()) {
                    outFile.mkdirs();
                } else {
                    outFile.getParentFile().mkdirs();
                    try (BufferedOutputStream bos =
                             new BufferedOutputStream(new FileOutputStream(outFile), BUFFER)) {
                        byte[] buf = new byte[BUFFER];
                        int len;
                        while ((len = zis.read(buf)) > 0) {
                            bos.write(buf, 0, len);
                        }
                    }
                }
                zis.closeEntry();
            }
        }
    }
}
"""

print("=" * 60)
print("TEST 4 — REFERENCE FIXED CODE")
print("=" * 60)
print(TEST4_FIXED)

result4 = analyze(TEST4_VULNERABLE, "TEST 4 | Path Traversal: Zip Slip — OOD Sub-Type")

---
## 🔴 TEST 5 — Insecure Deserialization (OOD: severely underrepresented)

**Why this is OOD**: Only **26 samples** in the entire 16K training set. The model has extremely limited exposure to this pattern.

**Expected**: VULNERABLE → Insecure Deserialization

**If model fails**: Very likely — this is the weakest category in training data.

**Fixed version**: Use an `ObjectInputStream` subclass that filters classes via `resolveClass()` allowlist.

In [ ]:
# ─── VULNERABLE CODE ─────────────────────────────────────────────────────────
TEST5_VULNERABLE = """
import java.io.*;
import java.util.Base64;

public class SessionManager {

    /**
     * Restores a user session from a Base64-encoded cookie value.
     * Vulnerable: deserialises arbitrary bytes from user-controlled input
     * — classic gadget-chain remote code execution vector.
     */
    public Object restoreSession(String cookieValue) throws Exception {
        byte[] data = Base64.getDecoder().decode(cookieValue);  // user-controlled!
        try (ObjectInputStream ois = new ObjectInputStream(
                new ByteArrayInputStream(data))) {
            return ois.readObject();  // No class filtering / no allowlist
        }
    }
}
"""

# ─── FIXED CODE (reference) ───────────────────────────────────────────────────
TEST5_FIXED = """
import java.io.*;
import java.util.Base64;
import java.util.HashSet;
import java.util.Set;

public class SessionManager {

    // FIXED: Create a resolveClass-filtered ObjectInputStream (allowlist approach)
    private static class SafeObjectInputStream extends ObjectInputStream {
        private static final Set<String> ALLOWED = new HashSet<>();
        static {
            // Only allow your own session class — block everything else
            ALLOWED.add("com.example.UserSession");
        }

        SafeObjectInputStream(InputStream in) throws IOException { super(in); }

        @Override
        protected Class<?> resolveClass(ObjectStreamClass desc)
                throws IOException, ClassNotFoundException {
            if (!ALLOWED.contains(desc.getName())) {
                throw new InvalidClassException("Blocked class: ", desc.getName());
            }
            return super.resolveClass(desc);
        }
    }

    public Object restoreSession(String cookieValue) throws Exception {
        byte[] data = Base64.getDecoder().decode(cookieValue);
        // FIXED: Use allowlist-filtered deserialisation
        try (SafeObjectInputStream ois = new SafeObjectInputStream(
                new ByteArrayInputStream(data))) {
            return ois.readObject();
        }
    }
}
"""

print("=" * 60)
print("TEST 5 — REFERENCE FIXED CODE")
print("=" * 60)
print(TEST5_FIXED)

result5 = analyze(TEST5_VULNERABLE, "TEST 5 | Insecure Deserialization — Severely Underrepresented (26 samples)")

---
## 🔴 TEST 6 — Subtle SQL Injection (multi-param dynamic WHERE)

This is the **original Custom_Test.ipynb** case — kept here for comparison with the OOD tests.

**Expected**: VULNERABLE → SQL Injection

**This should PASS** — it's close to training distribution.

In [ ]:
TEST6_SUBTLE_SQL = """
import java.sql.Connection;
import java.sql.PreparedStatement;
import java.sql.ResultSet;
import java.sql.SQLException;
import java.util.logging.Logger;

public class CustomerSearchService {

    private static final Logger logger = Logger.getLogger(CustomerSearchService.class.getName());

    private final Connection connection;

    public CustomerSearchService(Connection connection) {
        this.connection = connection;
    }

    public ResultSet searchCustomers(String nameFilter, String region) throws SQLException {

        logger.info("Searching customers with filter: " + nameFilter + ", region: " + region);

        // Subtle SQL Injection: dynamic query building in WHERE clause
        String query = "SELECT id, name, email, region FROM customers WHERE 1=1 ";

        if (nameFilter != null && !nameFilter.isEmpty()) {
            query += " AND name LIKE '%" + nameFilter + "%' ";
        }
        if (region != null && !region.isEmpty()) {
            query += " AND region = '" + region + "' ";
        }
        PreparedStatement stmt = connection.prepareStatement(query);
        return stmt.executeQuery();
    }
}
"""

result6 = analyze(TEST6_SUBTLE_SQL, "TEST 6 | Subtle SQL Injection (original example — should PASS)")

---
## ✅ TEST 7 — CLEAN Safe Code (False Positive check)

**Why important**: The model should NOT flag safe code as vulnerable. False positives make the tool unusable in production.

**Expected**: SECURE — no changes required

**If model fails**: It flags safe code as vulnerable (hallucinating a vulnerability).

In [ ]:
TEST7_SAFE = """
import java.sql.Connection;
import java.sql.PreparedStatement;
import java.sql.ResultSet;
import java.sql.SQLException;
import java.util.ArrayList;
import java.util.List;

/**
 * This class is intentionally SAFE — fully parameterised queries,
 * no user data in SQL string, proper resource handling with try-with-resources.
 */
public class OrderRepository {

    private final Connection conn;

    public OrderRepository(Connection conn) {
        this.conn = conn;
    }

    public List<String> getOrdersByUser(int userId) throws SQLException {
        String sql = "SELECT order_id FROM orders WHERE user_id = ?";
        List<String> orders = new ArrayList<>();

        try (PreparedStatement ps = conn.prepareStatement(sql)) {
            ps.setInt(1, userId);
            try (ResultSet rs = ps.executeQuery()) {
                while (rs.next()) {
                    orders.add(rs.getString("order_id"));
                }
            }
        }
        return orders;
    }

    public boolean cancelOrder(int orderId, int userId) throws SQLException {
        String sql = "UPDATE orders SET status = 'CANCELLED' WHERE order_id = ? AND user_id = ?";
        try (PreparedStatement ps = conn.prepareStatement(sql)) {
            ps.setInt(1, orderId);
            ps.setInt(2, userId);
            return ps.executeUpdate() > 0;
        }
    }
}
"""

result7 = analyze(TEST7_SAFE, "TEST 7 | CLEAN SAFE CODE — False Positive Check (expected: SECURE)")

---
## 🔴 TEST 8 — Multi-Vulnerability: SQL Injection + Log Injection (OOD: combo class)

**Why this is OOD**: Training examples had **one vulnerability per class**. This class contains **two** security issues simultaneously — the model must identify both.

**Expected**: VULNERABLE → SQL Injection + Log Injection

**If model fails**: May catch one but miss the other (likely misses Log Injection).

**Fixed version**: PreparedStatement for SQL Injection + sanitise newline chars before logging.

In [ ]:
# ─── VULNERABLE CODE ─────────────────────────────────────────────────────────
TEST8_MULTI_VULN = """
import java.sql.*;
import java.util.logging.Logger;

public class LoginService {

    private static final Logger LOG = Logger.getLogger(LoginService.class.getName());
    private final Connection conn;

    public LoginService(Connection conn) {
        this.conn = conn;
    }

    public boolean authenticate(String username, String password) throws SQLException {
        // VULNERABILITY 1 — SQL Injection: raw string concatenation into query
        String sql = "SELECT id FROM users WHERE username='" + username +
                     "' AND password_hash='" + password + "'";

        // VULNERABILITY 2 — Log Injection: newline characters in username can forge log entries
        // e.g. username = "admin\nINFO: Successful login for admin"
        LOG.info("Login attempt for user: " + username);

        try (Statement stmt = conn.createStatement();
             ResultSet rs   = stmt.executeQuery(sql)) {
            return rs.next();
        }
    }
}
"""

# ─── FIXED CODE (reference) ───────────────────────────────────────────────────
TEST8_FIXED = """
import java.sql.*;
import java.util.logging.Logger;

public class LoginService {

    private static final Logger LOG = Logger.getLogger(LoginService.class.getName());
    private final Connection conn;

    public LoginService(Connection conn) {
        this.conn = conn;
    }

    // FIXED for Log Injection: strip newline/carriage-return characters from log input
    private static String sanitiseForLog(String value) {
        if (value == null) return "(null)";
        return value.replaceAll("[\\r\\n\\t]", "_");
    }

    public boolean authenticate(String username, String password) throws SQLException {
        // FIXED for SQL Injection: use PreparedStatement with parameterised placeholders
        String sql = "SELECT id FROM users WHERE username = ? AND password_hash = ?";

        // FIXED for Log Injection: sanitise before logging
        LOG.info("Login attempt for user: " + sanitiseForLog(username));

        try (PreparedStatement ps = conn.prepareStatement(sql)) {
            ps.setString(1, username);
            ps.setString(2, password);
            try (ResultSet rs = ps.executeQuery()) {
                return rs.next();
            }
        }
    }
}
"""

print("=" * 60)
print("TEST 8 — REFERENCE FIXED CODE")
print("=" * 60)
print(TEST8_FIXED)

result8 = analyze(TEST8_MULTI_VULN, "TEST 8 | Multi-Vuln: SQL Injection + Log Injection combo — OOD")

---
## 📊 CELL 9 — Results Scorecard

In [ ]:
# ═══ CELL 9: Summary Scorecard ══════════════════════════════════════════════

SAFE_KEYWORDS = [
    "completely secure", "no vulnerability", "already safe",
    "no changes are required", "no vulnerabilities", "is safe", "is secure"
]

def says_vulnerable(text):
    return "VULNERABLE" in text.upper()

def says_secure(text):
    return any(kw in text.lower() for kw in SAFE_KEYWORDS)

tests = [
    {"id": 1, "name": "SQL Injection (String.format)",       "expected": "VULNERABLE", "result": result1, "why_ood": "String.format() vs + concat"},
    {"id": 2, "name": "XSS in REST API JSON response",       "expected": "VULNERABLE", "result": result2, "why_ood": "JSON body vs HTML rendering"},
    {"id": 3, "name": "Command Injection (ProcessBuilder)",  "expected": "VULNERABLE", "result": result3, "why_ood": "ProcessBuilder+sh vs Runtime.exec"},
    {"id": 4, "name": "Path Traversal (Zip Slip)",           "expected": "VULNERABLE", "result": result4, "why_ood": "Only 104 samples in training"},
    {"id": 5, "name": "Insecure Deserialization",            "expected": "VULNERABLE", "result": result5, "why_ood": "Only 26 samples in training"},
    {"id": 6, "name": "Subtle SQL (dynamic WHERE)",          "expected": "VULNERABLE", "result": result6, "why_ood": "In-distribution (baseline)"},
    {"id": 7, "name": "CLEAN safe code",                     "expected": "SECURE",    "result": result7, "why_ood": "False-positive check"},
    {"id": 8, "name": "Multi-Vuln (SQL + Log Injection)",    "expected": "VULNERABLE", "result": result8, "why_ood": "Two vulns in one class"},
]

print("\n" + "=" * 78)
print("  OOD TEST SCORECARD — 16K Model (java-vuln-adapter-32b-full)")
print("=" * 78)
print(f"  {'#':<3} {'Test':<40} {'Expected':<11} {'Model':<11} {'Pass?'}")
print("-" * 78)

passed = 0
failed_tests = []

for t in tests:
    r = t["result"]
    if says_vulnerable(r):
        model_says = "VULNERABLE"
    elif says_secure(r):
        model_says = "SECURE"
    else:
        model_says = "UNCLEAR"

    correct = model_says == t["expected"]
    if correct:
        passed += 1
        status = "PASS"
    else:
        failed_tests.append(t)
        status = "FAIL"
    print(f"  {t['id']:<3} {t['name']:<40} {t['expected']:<11} {model_says:<11} {status}")

print("=" * 78)
print(f"  SCORE: {passed}/8  ({passed/8*100:.0f}%)")

if failed_tests:
    print(f"\n  FAILED TESTS & ROOT CAUSE:")
    for t in failed_tests:
        print(f"    Test #{t['id']} — {t['name']}")
        print(f"             Reason: {t['why_ood']}")
    print("\n  Recommendation: add these patterns to the next fine-tune iteration.")
else:
    print("  All OOD tests passed — model generalises well!")
print("=" * 78)